# Consuming the AIFaultBench Benchmark

This notebook is a **practical companion** to the dataset. It shows you how to:

1. **Load & explore** the index (what's in the benchmark, how it's shaped, and its six domains).
2. **Read the structure** — the anatomy of one bug folder, and which parts are a *stable contract* vs. free-form.
3. **Inspect a bug end-to-end** using the *evidence already shipped in the package* (no execution required).
4. **Reproduce a bug for real** — an opt-in, network-touching runner (clone → venv → run).
5. **Filter & select** bugs for experiments (by domain, repository, reproducibility, blocking reason).
6. **Use it as an oracle** for automated tools — APR / repair systems, bug-reproducer agents, fault localizers.
7. **Emit an "agent task packet"** — a compact, self-contained JSON bundle any agent can ingest per bug.

---
## 0. Locate the dataset

The notebook auto-detects the dataset root by walking up from the current directory until it finds 
`index.json`. If you moved the notebook, set `DATASET_ROOT` by hand.

In [ ]:
from __future__ import annotations  # so type hints work on Python 3.8/3.9 too
import json, csv, collections, os, subprocess, textwrap
from pathlib import Path

def find_dataset_root(start: Path | None = None) -> Path:
    """Walk upward from `start` (default: cwd) until index.json is found."""
    here = (start or Path.cwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / 'index.json').exists() and (cand / 'bugs').is_dir():
            return cand
    raise FileNotFoundError(
        'Could not locate the dataset root (a folder containing index.json and bugs/). '
        'Set DATASET_ROOT manually.'
    )

DATASET_ROOT = find_dataset_root()
BUGS_DIR = DATASET_ROOT / 'bugs'
print('Dataset root :', DATASET_ROOT)
print('Bug folders  :', sum(1 for p in BUGS_DIR.iterdir() if p.is_dir()))

---
## 1. Load & explore the index

`index.json` is the **canonical, source-of-truth map**: one record per bug. Prefer it over `index.csv` 
for programmatic use — see the boxed warning below for why.

In [ ]:
BUGS = json.loads((DATASET_ROOT / 'index.json').read_text())
print(f'{len(BUGS)} bugs loaded from index.json')
print('\nOne record (fields are the stable index schema):')
print(json.dumps(BUGS[0], indent=2))

### ⚠️ Use `index.json`, not `index.csv`, for filtering

The two files describe the same 770 bugs but are **not interchangeable**:

- **`index.json` is canonical for reproduction.** It alone carries `commit`, `clone_url`,
  `bug_report`, `codebase_present`, `blocking_reason`, `setup_command`, `run_command`, and the
  nested `github` object. `reproducible` and `codebase_present` are native JSON **booleans**.
- **`index.csv` is the flat mining view.** It alone carries `issue_state`, `author`, `n_labels`,
  `labels`, `n_comments`, `has_fix`, `linked_pr_numbers`, `enrich_error`. Every value is a
  **string**.

There are *two* silent-filter traps in the CSV, and the second one bites even careful readers:

1. Every non-empty string is truthy, so `[r for r in rows if r['reproducible']]` returns **all 770**.
2. The booleans are serialized **Python-style** — `"True"` / `"False"`, capitalized. Comparing
   against the lowercase `== 'true'` matches **nothing** and silently returns an empty set.

The cell below demonstrates all three behaviours side by side.


In [ ]:
# JSON: native booleans -> the truthy idiom is correct
json_repro = [b for b in BUGS if b['reproducible']]
print('index.json  ->  reproducible via truthy idiom :', len(json_repro))

# CSV: everything is a string, and booleans are Python-cased ("True"/"False")
csv_rows = list(csv.DictReader((DATASET_ROOT / 'index.csv').open()))
print('index.csv   ->  raw values seen              :',
      sorted({r['reproducible'] for r in csv_rows}))

csv_truthy = [r for r in csv_rows if r['reproducible']]              # WRONG: all strings truthy
csv_lower  = [r for r in csv_rows if r['reproducible'] == 'true']    # WRONG: values are 'True'
csv_right  = [r for r in csv_rows if r['reproducible'] == 'True']    # correct
print('index.csv   ->  truthy idiom      (WRONG)    :', len(csv_truthy))
print("index.csv   ->  == 'true'         (WRONG)    :", len(csv_lower))
print("index.csv   ->  == 'True'         (correct)  :", len(csv_right))

assert len(json_repro) == len(csv_right), 'JSON and CSV should agree once compared correctly'
assert len(csv_lower) == 0, 'the lowercase comparison silently matches nothing'
print('\nRule of thumb: load index.json and rely on native booleans.')


### The shape of the benchmark

A few quick aggregations over the index — all pure stdlib.

In [ ]:
total = len(BUGS)
repro = sum(b['reproducible'] for b in BUGS)
print(f'Bugs             : {total}')
print(f'Reproducible     : {repro} / {total}  ({repro/total:.0%})')
print(f'Not reproducible : {total - repro} / {total}  ({(total-repro)/total:.0%})')

repos = collections.Counter(b['repository'] for b in BUGS)
orgs = {b['repository'].split('/')[0] for b in BUGS}
print(f'Repositories     : {len(repos)}   across {len(orgs)} organizations')

print('\nTop 10 repositories by bug count:')
for name, n in repos.most_common(10):
    rr = sum(b['reproducible'] for b in BUGS if b['repository'] == name)
    bar = '#' * round(30 * n / repos.most_common(1)[0][1])
    print(f'  {n:3d}  {bar:<30}  {name}   ({rr}/{n} repro)')


> **Group by `repository`, not `library`.** The `library` column is a free-text mining label with 
> casing duplicates (`SDV` vs `sdv`) and, in one row, a full `owner/repo` pasted in. The `repository` 
> field is the clean grouping key. The cell below shows the discrepancy.

In [ ]:
libs = sorted({b['library'] for b in BUGS})
repos_set = {b['repository'] for b in BUGS}
print(f'distinct `library` labels   : {len(libs)}')
print(f'distinct `repository` values: {len(repos_set)}   <- use this one')

# 1) a full owner/repo string pasted into the library column:
slash = sorted(l for l in libs if '/' in l)
print('library values containing "/" (should be a repo, not a library):', slash)

# 2) case-duplicate labels that collapse to the same library:
folded = collections.defaultdict(set)
for l in libs:
    folded[l.lower()].add(l)
case_dups = {k: sorted(v) for k, v in folded.items() if len(v) > 1}
print('case-duplicate labels (same library, different casing):', case_dups)

### The six domains

Every bug carries a `domain` label — the one field that tells you *what kind of system* the fault
lives in. It is the recommended stratification axis: the domains differ sharply in reproduction
rate, and a single aggregate number hides most of the signal.

The mapping from repository to domain is explicit and re-runnable in `scripts/assign_domain.py`.


In [ ]:
ORDER = ['DL', 'LLM', 'Agentic', 'ML', 'RL', 'Tooling']
print(f"{'domain':10s} {'bugs':>5s} {'repos':>6s} {'repro':>6s} {'rate':>6s}")
for d in ORDER:
    sub = [b for b in BUGS if b['domain'] == d]
    r = sum(b['reproducible'] for b in sub)
    nrepos = len({b['repository'] for b in sub})
    print(f'{d:10s} {len(sub):5d} {nrepos:6d} {r:6d} {r/len(sub):6.0%}')

assert sum(1 for b in BUGS if b['domain'] in ORDER) == len(BUGS), 'unknown domain label'
print('\nAgentic repositories:')
ag = collections.Counter(b['repository'] for b in BUGS if b['domain'] == 'Agentic')
for name, n in ag.most_common():
    rr = sum(b['reproducible'] for b in BUGS
             if b['repository'] == name and b['domain'] == 'Agentic')
    print(f'  {n:3d}  {name:36s} ({rr}/{n} repro)')


> **Why `Agentic` reproduces highest (92%).** Agent-framework faults concentrate in orchestration,
> serialization, schema and tool-call handling, and state management. Those run on CPU and can be
> triggered against *recorded* payloads — no provider client, no credential, no network call. Faults
> in `LLM` serving and training stacks (75%) are far more often gated on a specific accelerator.
> If you want a CPU-only evaluation set, the 119 reproducible agentic bugs are the cheapest entry point.


---
## 2. The anatomy of a bug folder

Every bug lives in `bugs/<bug_id>/` (zero-padded: `001` … `774`, 770 folders). **All ten files ship for all 770 bugs** 
— that's the reliable per-bug contract:

| file | purpose | format |
|------|---------|--------|
| `bug_report.txt` | the original GitHub issue (title, URL, body, repro steps) | text |
| `repro.py` | minimal script that triggers the bug | Python |
| `requirements.txt` | bug-specific Python dependencies | pip |
| `setup_codebase.sh` | `git clone` + `git checkout <commit>` — recreates the source | bash |
| `setup_env.sh` | creates a local `.venv`, installs `requirements.txt` | bash |
| `run_repro.sh` | runs `setup_env.sh` then `repro.py`, capturing output | bash |
| `reproduction.json` | **structured result** — the machine-readable oracle record | JSON (stable) |
| `reproduction_trajectory.md` | human narrative: how it was reproduced + what was seen | Markdown |
| `repro_stdout.log` / `repro_stderr.log` | captured output = the shipped **evidence** | text |
| `README.md`, `manifest.json` | per-bug notes & loose metadata | text / JSON |

Some folders carry **extra** helper modules, config, vendored `codebase/`, or input fixtures where a 
plain clone wasn't enough. Those are the only non-regenerable inputs and are kept deliberately.

In [ ]:
# The 10 canonical files. Unlike most benchmarks, these are present for EVERY bug — verify it.
CANONICAL = [
    'bug_report.txt', 'repro.py', 'requirements.txt', 'setup_codebase.sh',
    'setup_env.sh', 'run_repro.sh', 'reproduction.json',
    'reproduction_trajectory.md', 'README.md', 'manifest.json',
]
n_dirs = sum(1 for d in BUGS_DIR.iterdir() if d.is_dir())
present = collections.Counter()
for d in BUGS_DIR.iterdir():
    if not d.is_dir():
        continue
    for f in CANONICAL:
        present[f] += (d / f).exists()
for f in CANONICAL:
    print(f'{present[f]:4d}/{n_dirs}  {f}')

assert n_dirs == len(BUGS), 'a bug folder exists that the index does not list (or vice versa)'
assert all(present[f] == n_dirs for f in CANONICAL), 'the 10-file contract is broken'
print('\nContract holds: all 10 files present for all', n_dirs, 'bugs.')


### A tiny helper: the `Bug` accessor

Rather than remember filenames, wrap a bug folder in a small class. Everything downstream uses it.

In [ ]:
class Bug:
    """Convenience accessor for one bug folder + its index record."""
    _index = {b['bug_id']: b for b in BUGS}

    def __init__(self, bug_id: str):
        bug_id = str(bug_id).zfill(3)
        if bug_id not in self._index:
            raise KeyError(f'unknown bug_id {bug_id!r}')
        self.id = bug_id
        self.dir = BUGS_DIR / bug_id
        self.meta = self._index[bug_id]          # the index.json record

    # --- index shortcuts ---
    @property
    def repository(self):     return self.meta['repository']
    @property
    def issue_url(self):      return self.meta['issue_url']
    @property
    def commit(self):         return self.meta['commit']
    @property
    def domain(self):         return self.meta['domain']
    @property
    def reproducible(self):   return self.meta['reproducible']
    @property
    def blocking_reason(self): return self.meta['blocking_reason']

    # --- file readers (return '' if a file is absent) ---
    def _read(self, name):
        p = self.dir / name
        return p.read_text(errors='replace') if p.exists() else ''
    def report(self):     return self._read('bug_report.txt')
    def repro_code(self): return self._read('repro.py')
    def requirements(self): return self._read('requirements.txt')
    def trajectory(self): return self._read('reproduction_trajectory.md')
    def stdout(self):     return self._read('repro_stdout.log')
    def stderr(self):     return self._read('repro_stderr.log')
    def reproduction(self):
        """The structured reproduction.json record (the stable 5-key schema)."""
        raw = self._read('reproduction.json')
        return json.loads(raw) if raw else {}
    def files(self):
        """Every file shipped in this bug folder (incl. any extras)."""
        return sorted(p.name for p in self.dir.iterdir())

    def __repr__(self):
        flag = 'REPRO' if self.reproducible else 'blocked'
        return f'<Bug {self.id} {self.domain} {self.repository} [{flag}]>'

# Two running examples: one deep-learning bug, one agentic bug. Everything
# downstream works identically on both -- that is the point of the merge.
example = next(Bug(b['bug_id']) for b in BUGS if b['reproducible'] and b['domain'] == 'DL')
example_agentic = next(Bug(b['bug_id']) for b in BUGS
                       if b['reproducible'] and b['domain'] == 'Agentic')
example, example_agentic

---
## 3. Inspect a bug end-to-end (using shipped evidence)

You can understand a bug **completely without running anything** — the package ships the reproduction 
evidence. Let's walk through our example bug.

In [ ]:
b = example
print('BUG', b.id, '—', b.repository, f'[{b.domain}]')
print('issue :', b.issue_url)
print('commit:', b.commit)
print('files :', b.files())


**3a. The bug report** (the original GitHub issue, verbatim):

In [ ]:
print(b.report()[:1200])

**3b. The minimal reproduction script** — a self-contained trigger for the bug:

In [ ]:
print(b.repro_code()[:1600])

**3c. The structured oracle record — `reproduction.json`.** This is the *one JSON schema you can rely 
on across all 770 bugs*: exactly these five keys, always present.

In [ ]:
rep = b.reproduction()
print('keys:', list(rep))
print()
print(json.dumps(rep, indent=2)[:1400])

In [ ]:
# Confirm the 5-key schema is stable across the WHOLE dataset.
key_sets = collections.Counter(
    tuple(sorted(Bug(x['bug_id']).reproduction().keys())) for x in BUGS
)
for keys, n in key_sets.most_common():
    print(f'{n:4d} bugs  <-  {keys}')

**3d. The shipped evidence** — captured stdout from the actual reproduction run. This is what "the bug 
reproducing" looked like on the reference machine, with **zero execution on your part**.

In [ ]:
out = b.stdout()
print(f'repro_stdout.log  ({len(out)} chars) — tail:')
print('-' * 60)
print(out[-800:] if out else '(empty)')

> **`manifest.json` is *not* a contract.** It's free-form per-bug metadata — across the dataset it uses 
> 100+ different key names with inconsistent presence. Read it for a human hint if you like, but **do not 
> build tooling that parses it**. The stable structured surfaces are `index.json` and `reproduction.json`.

---
## 4. Reproduce a bug for real (opt-in)

Everything above used shipped evidence. To reproduce a bug **live**, the flow is:

```
bash setup_codebase.sh   # git clone + checkout the buggy commit  (network)
bash run_repro.sh        # create .venv, pip install, run repro.py (network + build)
```

This **clones repositories, creates a virtualenv, and installs packages** — it touches the network, can 
take minutes, and some bugs need a specific Python interpreter or a GPU. So it is **disabled by default**.
Flip the flag below to run it for a single, CPU-friendly bug.

In [ ]:
# --- SAFETY FLAG: leave False unless you intend to clone + install + execute. ---
RUN_LIVE_REPRODUCTION = False

def reproduce_live(bug: 'Bug', timeout_setup=1200, timeout_run=1800):
    """Run setup_codebase.sh then run_repro.sh in the bug folder. Returns a dict."""
    def sh(script, timeout):
        proc = subprocess.run(
            ['bash', script], cwd=bug.dir, timeout=timeout,
            capture_output=True, text=True,
        )
        return proc.returncode, proc.stdout, proc.stderr
    result = {'bug_id': bug.id}
    result['setup_rc'], _, result['setup_err'] = sh('setup_codebase.sh', timeout_setup)
    result['run_rc'], result['run_out'], result['run_err'] = sh('run_repro.sh', timeout_run)
    return result

if RUN_LIVE_REPRODUCTION:
    res = reproduce_live(example)
    print('setup_codebase.sh rc:', res['setup_rc'])
    print('run_repro.sh      rc:', res['run_rc'])
    print('--- run stdout (tail) ---')
    print(res['run_out'][-800:])
else:
    print('Live reproduction is OFF. Set RUN_LIVE_REPRODUCTION = True to clone + install + run.')
    print('Meanwhile, the shipped evidence in Section 3d shows the expected outcome.')

> **Isolation matters.** Dependency sets across bugs *conflict by design* (different library versions, 
> CUDA builds, Python versions). `setup_env.sh` already isolates each bug in its own `.venv/`. Run each 
> bug in a fresh environment; don't share one interpreter across bugs.

> **Interpreter pinning.** A bug's `setup_env.sh` may request a specific interpreter (e.g. `python3.10`, 
> `python3.11`). Install that interpreter if it isn't on `PATH`. Most bugs use the default `python3`.

---
## 5. Filter & select bugs for experiments

Common selection recipes over the index. These are the building blocks for a benchmark run.

In [ ]:
# (a) The clean pass/fail set.
reproducible = [Bug(x['bug_id']) for x in BUGS if x['reproducible']]
print('reproducible bugs        :', len(reproducible))

# (b) By domain -- the recommended stratification axis.
agentic = [Bug(x['bug_id']) for x in BUGS if x['domain'] == 'Agentic']
agentic_ok = [b for b in agentic if b.reproducible]
print('agentic bugs             :', len(agentic), f'({len(agentic_ok)} reproducible)')

# (c) Everything from one repository.
transformers = [Bug(x['bug_id']) for x in BUGS if x['repository'] == 'huggingface/transformers']
print('huggingface/transformers :', len(transformers))

# (d) Group bug_ids by repository (the recommended grouping key).
by_repo = collections.defaultdict(list)
for x in BUGS:
    by_repo[x['repository']].append(x['bug_id'])
print('repositories             :', len(by_repo))

# (e) The non-reproducible study set (116 of 118 carry a blocking_reason).
blocked = [Bug(x['bug_id']) for x in BUGS if not x['reproducible']]
unlabeled = [b for b in blocked if not b.blocking_reason]
print('blocked                  :', len(blocked),
      f'({len(blocked) - len(unlabeled)} labeled, {len(unlabeled)} without a reason:',
      ', '.join(b.id for b in unlabeled) + ')')

# (f) A CPU-only, key-free evaluation slice.
cpu_friendly = [b for b in agentic_ok]
print('CPU-only agentic slice   :', len(cpu_friendly))

print('\nexample blocking_reason:')
print(textwrap.fill(next(b for b in blocked if b.blocking_reason).blocking_reason, 92))


### Approximate categorization of the 118 blocked bugs

`blocking_reason` is **free text**, not a categorical field. The README groups it coarsely into
*upstream-fix / drift* (~69), *hardware / OS / build* (~45), and *not-an-executable-bug* (~4) — a
**manual** classification that is hardcoded in `scripts/make_figures.py`, not derived from the index.
Below is a **keyword heuristic** that *approximates* that grouping so you can slice the set yourself.
Treat the counts as indicative, **not** as an exact recovery of the README's 69 / 45 / 4.

The heuristic leaves ~10 cases `uncategorized` rather than guessing — including the two blocked
bugs (`157`, `541`) that carry an *empty* `blocking_reason`. That gap is deliberate: the manual
grouping used judgment the keywords cannot recover.


In [ ]:
def categorize_blocking(reason: str) -> str:
    """Approximate the README's manual grouping via keywords over free text.

    Order matters: an unambiguous hardware/build signal (CUDA, GPU, NCCL, a pinned
    wheel) is treated as hardware-gated even when the reason ALSO says 'does not
    reproduce here' -- most blocked reasons share that phrasing regardless of cause.
    """
    r = reason.lower()
    noncode = ('documentation', 'doc-only', 'tagging', 'not an executable',
               'non-code', 'feature request', 'readme file')
    hardware = ('gpu', 'cuda', 'nccl', 'accelerator', 'h100', 'hopper', 'rocm', ' mps ',
                'multi-gpu', 'multiple gpu', 'hardware', 'nvidia', 'tpu', ' ampere',
                'triton', 'bitsandbytes', 'nightly wheel', 'pinned build',
                'specific build', 'operating system')
    drift = ('no longer', 'not reproducible', 'cannot reproduce', 'could not reproduce',
             'did not reproduce', 'does not reproduce', 'not reproduce', 'no error',
             'behaves as expected', 'as expected', 'works', 'passes', 'succeeds',
             'imports successfully', 'checkout', 'fixed', 'upstream', 'drift', 'flaky',
             'nondetermin', 'resolved', 'does not occur', 'did not occur', 'not occur',
             'no failure', 'already', 'reported behavior', 'reported failure',
             'does not raise', 'not reachable', 'not runnable')
    if any(k in r for k in noncode):  return 'not-an-executable-bug'
    if any(k in r for k in hardware): return 'hardware / OS / build'
    if any(k in r for k in drift):    return 'upstream-fix / drift'
    return 'uncategorized'

blocked_rows = [x for x in BUGS if not x['reproducible']]
cats = collections.Counter(categorize_blocking(x['blocking_reason']) for x in blocked_rows)
print(f'Heuristic grouping of the {len(blocked_rows)} blocked bugs (APPROXIMATE — free-text keywords):')
for cat, n in cats.most_common():
    print(f'  {n:3d}  {cat}')
print('\n(README manual grouping, for reference: ~69 upstream-fix/drift, ~45 hardware, ~4 non-code.')
print(' The ranking matches; exact per-bug lines are judgment-dependent, so treat counts as indicative.)')

print('\nBlocked bugs by domain:')
for d, n in collections.Counter(x['domain'] for x in blocked_rows).most_common():
    print(f'  {n:3d}  {d}')

### ⚠️ Before you use it as an APR oracle: filter the mirror-style reproductions

Reproduction depth is not uniform. **44 of the 652 reproducible scripts re-implement the buggy
logic locally** rather than driving the subject library, and **13 of those never import the library
or touch `codebase/` at all**. Those 13 still *demonstrate* the fault faithfully — but a fix applied
to the upstream project cannot change their outcome, so **they are not valid repair targets** and
will register as "unfixed" no matter what your tool does.

The record ships in `scripts/mirror_style_repros.json` (40 entries; four more — `046`, `347`, `354`,
`589` — carry the disclosure as an inline comment in `repro.py`). Build your evaluation set with it.


In [ ]:
MIRROR_PATH = DATASET_ROOT / 'scripts' / 'mirror_style_repros.json'
INLINE_DISCLOSED = {'046', '347', '354', '589'}   # same disclosure, as an inline comment

mirror = json.loads(MIRROR_PATH.read_text()) if MIRROR_PATH.exists() else []
not_repair_targets = {m['bug_id'] for m in mirror if not m['touches_subject_library']}
mirror_style = {m['bug_id'] for m in mirror} | INLINE_DISCLOSED

print(f'mirror-style scripts recorded : {len(mirror_style)}')
print(f'  ...of which never touch the subject library (NOT repair targets): {len(not_repair_targets)}')
print('  ', sorted(not_repair_targets))

# The set you actually want for repair / APR evaluation:
apr_set = [b for b in reproducible if b.id not in not_repair_targets]
print(f'\nreproducible bugs              : {len(reproducible)}')
print(f'valid APR targets              : {len(apr_set)}')

# Same slice, restricted to the CPU-only agentic domain:
apr_agentic = [b for b in apr_set if b.domain == 'Agentic']
print(f'valid APR targets, Agentic only: {len(apr_agentic)}')


---
## 6. Use the benchmark as an oracle (APR / repair / agent evaluation)

For evaluating a bug-fixing tool, the contract is simple:

> A bug is **fixed** when its reproduction **no longer produces the reported signal**.

Operationally, the oracle is: run `run_repro.sh`; on an *unfixed* codebase it triggers the bug 
(non-zero exit / reported error). Apply your candidate fix, run it again; if it now passes, the fix 
holds. The skeleton below shows the loop — **guarded by the same live flag** so it doesn't execute by 
default.

In [ ]:
def evaluate_fix(bug: 'Bug', apply_fix, *, live=False):
    """Skeleton APR/agent evaluation harness for one bug.

    apply_fix(codebase_dir) -> None : your tool mutates the checked-out codebase in place.
    Returns a verdict dict. With live=False it only *plans* the steps (dry run).
    """
    plan = [
        f'cd {bug.dir}',
        'bash setup_codebase.sh        # clone + checkout buggy commit',
        'bash run_repro.sh   # BEFORE fix: expect the bug to trigger (oracle = fail)',
        '<apply_fix to codebase/>      # your APR system / agent edits the source',
        'bash run_repro.sh   # AFTER fix: expect it to pass (oracle = no signal)',
    ]
    if not live:
        return {'bug_id': bug.id, 'dry_run': True, 'plan': plan}

    def run():
        p = subprocess.run(['bash', 'run_repro.sh'], cwd=bug.dir,
                           capture_output=True, text=True, timeout=1800)
        return p.returncode
    subprocess.run(['bash', 'setup_codebase.sh'], cwd=bug.dir, timeout=1200)
    before = run()                       # expect non-zero (bug triggers)
    apply_fix(bug.dir / 'codebase')      # your tool edits the source in place
    after = run()                        # expect zero (bug gone)
    return {'bug_id': bug.id, 'triggered_before': before != 0,
            'passes_after': after == 0,
            'fixed': before != 0 and after == 0}

# Dry run — shows the exact oracle steps without executing anything.
verdict = evaluate_fix(example, apply_fix=lambda cb: None, live=False)
print(json.dumps(verdict, indent=2))

**A full-benchmark run** is just this loop over the reproducible set, each bug in its own fresh 
environment, resources pinned. Sketch (kept as a string so nothing runs):

In [ ]:
harness_sketch = '''
results = []
for bug in apr_set:                      # valid repair targets only
    # isolate: each bug owns its .venv (setup_env.sh handles this)
    v = evaluate_fix(bug, apply_fix=my_repair_tool, live=True)
    results.append(v)
    print(bug.id, 'FIXED' if v['fixed'] else 'unfixed')

passed = sum(r['fixed'] for r in results)
print(f'repair rate: {passed}/{len(results)} = {passed/len(results):.1%}')
'''
print(harness_sketch)
print('Tips: run bugs in parallel but PIN cpu/gpu (some are memory- or GPU-hungry);')
print('start from apr_set (mirror-style scripts filtered out);')
print('use the 118 blocked bugs for environment-sensitivity study; stratify results by domain.')

---
## 7. Emit an "agent task packet" for other agents

To feed a bug to an autonomous agent (LLM repair agent, reproducer, fault localizer), you want a 
**single self-contained bundle** it can ingest — the report, the reproduction, the exact commands to 
recreate the codebase, the oracle, and the shipped evidence. The function below assembles that from the 
stable surfaces only (`index.json` + the canonical files + `reproduction.json`).

In [ ]:
def agent_task_packet(bug: 'Bug', *, include_evidence=True, max_log=2000) -> dict:
    """Build a self-contained JSON packet describing one bug for an agent."""
    rep = bug.reproduction()
    packet = {
        'bug_id': bug.id,
        'repository': bug.repository,
        'domain': bug.domain,
        'issue_url': bug.issue_url,
        'commit': bug.commit,
        'clone_url': bug.meta['clone_url'],
        'reproducible': bug.reproducible,
        'blocking_reason': bug.blocking_reason or None,
        # --- what the agent should read ---
        'bug_report': bug.report(),
        'repro_script': bug.repro_code(),
        'requirements': bug.requirements(),
        # --- how to recreate + run (commands, not opaque state) ---
        'setup': {
            'recreate_codebase': bug.meta['setup_command'],
            'run_reproduction': bug.meta['run_command'],
            'codebase_ships_in_folder': bug.meta['codebase_present'],
        },
        # --- the oracle the agent is graded against ---
        'oracle': {
            'definition': 'Bug is fixed when run_repro.sh no longer produces the reported signal.',
            'reproduction_command': rep.get('reproduction_command'),
            'expected_steps': rep.get('steps'),
            'reference_evidence': rep.get('evidence'),
            # False => the script does not drive the subject library, so an upstream
            # fix cannot change its outcome. Do not grade a repair tool on these.
            'valid_repair_target': bug.id not in not_repair_targets,
        },
        'extra_files': [f for f in bug.files() if f not in {
            'bug_report.txt','repro.py','requirements.txt','setup_codebase.sh','setup_env.sh',
            'run_repro.sh','reproduction.json','reproduction_trajectory.md','README.md',
            'manifest.json','repro_stdout.log','repro_stderr.log'}],
    }
    if include_evidence:
        packet['shipped_stdout'] = bug.stdout()[-max_log:]
        packet['shipped_stderr'] = bug.stderr()[-max_log:]
    return packet

packet = agent_task_packet(example)
print('packet keys:', list(packet))
print('\noracle block:')
print(json.dumps(packet['oracle'], indent=2)[:900])

# The identical call works on an agentic bug -- same schema, same contract.
ag_packet = agent_task_packet(example_agentic)
print('\nagentic packet:', ag_packet['bug_id'], ag_packet['repository'],
      f"[{ag_packet['domain']}]")
assert list(ag_packet) == list(packet), 'packet schema is uniform across domains'
print('packet schema identical across domains: True')

The packet is plain JSON — write one per bug to disk and any agent framework can pick them up:

In [ ]:
# Example: materialize packets for a small selection into the scratchpad.
import tempfile
outdir = Path(tempfile.mkdtemp(prefix='agent_packets_'))
selection = reproducible[:5]
for bug in selection:
    (outdir / f'{bug.id}.json').write_text(json.dumps(agent_task_packet(bug), indent=2))
written = sorted(p.name for p in outdir.iterdir())
print(f'wrote {len(written)} packets to {outdir}')
print(written)
# Sanity: each packet round-trips and carries the oracle + report.
sample = json.loads((outdir / written[0]).read_text())
assert sample['bug_report'] and sample['oracle']['definition']
print('\npacket size (chars):', len((outdir / written[0]).read_text()))

### A rendered prompt for an LLM repair agent

If you're driving an LLM directly, flatten the packet into a prompt. Here's a compact renderer — adapt 
the framing to your system.

In [ ]:
def render_agent_prompt(bug: 'Bug') -> str:
    p = agent_task_packet(bug, include_evidence=True, max_log=1200)
    return textwrap.dedent(f'''\
        You are debugging a real issue in {p['repository']} at commit {p['commit']}.

        ## Issue
        {p['issue_url']}
        {p['bug_report'][:1500]}

        ## Reproduction script (repro.py)
        ```python
        {p['repro_script'][:1500]}
        ```

        ## How to recreate the codebase and run the reproduction
        1. {p['setup']['recreate_codebase']}
        2. {p['setup']['run_reproduction']}

        ## Oracle (how your fix is judged)
        {p['oracle']['definition']}
        Reference evidence of the bug triggering:
        {p['oracle']['reference_evidence']}

        ## Your task
        Locate the root cause in the checked-out codebase and produce a patch so that
        the reproduction command no longer produces the reported signal.
        ''')

print(render_agent_prompt(example)[:1800])

---
## 8. Optional: pandas & a quick chart

Everything above is stdlib. If you *have* pandas / matplotlib, these give you a DataFrame and one chart. 
Both cells **skip cleanly** if the packages aren't installed. (The dataset also ships rendered figures in 
`assets/` and the script `scripts/make_figures.py`, so you rarely need to plot yourself.)

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(BUGS)
    summary = (df.groupby('repository')
                 .agg(bugs=('bug_id', 'size'), reproducible=('reproducible', 'sum'))
                 .assign(rate=lambda d: (d.reproducible / d.bugs).round(2))
                 .sort_values('bugs', ascending=False))
    print(summary.head(12).to_string())
except ImportError:
    print('pandas not installed — skipping. (The stdlib aggregations in Section 1 cover this.)')

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')  # headless-safe
    import matplotlib.pyplot as plt
    repo_counts = collections.Counter(b['repository'] for b in BUGS).most_common(10)
    names = [n for n, _ in repo_counts][::-1]
    repro_n = [sum(x['reproducible'] for x in BUGS if x['repository'] == n) for n in names]
    total_n = [c for _, c in repo_counts][::-1]
    blocked_n = [t - r for t, r in zip(total_n, repro_n)]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(names, repro_n, label='reproducible')
    ax.barh(names, blocked_n, left=repro_n, label='blocked')
    ax.set_xlabel('bugs'); ax.set_title('Top 10 repositories (reproducible vs blocked)')
    ax.legend(); fig.tight_layout()
    import tempfile
    out_png = Path(tempfile.gettempdir()) / 'dlbugs_top_repos.png'  # tempdir, not the repo
    fig.savefig(out_png, dpi=90)
    print('saved chart to', out_png)
except ImportError:
    print('matplotlib not installed — skipping. See assets/ for the shipped figures.')

---
## 9. Cheat-sheet

```python
# load (canonical, native booleans)
BUGS = json.loads(Path('index.json').read_text())      # 770 bugs, one flat bugs/ tree

# filter
repro   = [b for b in BUGS if b['reproducible']]              # 652 clean pass/fail
blocked = [b for b in BUGS if not b['reproducible']]          # 118 (116 carry a blocking_reason)
agentic = [b for b in BUGS if b['domain'] == 'Agentic']       # 130 (119 reproducible, CPU-only)
by_repo = ...  # group by b['repository']  (NOT b['library'])
# domains: DL 307 | LLM 151 | Agentic 130 | ML 118 | RL 40 | Tooling 24

# the CSV needs explicit, correctly-cased comparison
[r for r in csv.DictReader(open('index.csv')) if r['reproducible'] == 'True']   # not 'true'

# one bug
bug = Bug('001')                  # or Bug('645') for an agentic one
bug.report(); bug.repro_code(); bug.reproduction(); bug.stdout(); bug.domain

# reproduce live (opt-in, network + install)
#   bash setup_codebase.sh && bash run_repro.sh

# oracle:  fixed  <=>  run_repro.sh no longer produces the reported signal
#   BUT drop the 13 bugs in scripts/mirror_style_repros.json whose script never
#   touches the subject library -- an upstream fix cannot change their outcome.
# agent:   agent_task_packet(bug)  ->  self-contained JSON bundle
```

**Contracts to rely on:** `index.json` (native booleans) and each bug's `reproduction.json` (exactly
5 keys) + the 10 canonical files, all present for all 770 bugs. **Do not** parse `manifest.json`
(free-form), and do not filter the raw CSV without `== 'True'`. **Group by `repository`, stratify by
`domain`.**

See the dataset `README.md` for the full narrative, per-domain and per-repository reproduction rates,
the known limitations, and the citation.
